# RTC-NER-Extended Model Training and Evaluation

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
os.chdir('/content/drive/MyDrive/Patricia RTC NER Research/Data-in-brief/rtc_ner_ext_model')

In [ ]:
!ls

rtc_test_annotations_3.json  rtc_train_annotations_3.json


In [ ]:
#!python3 -m pip install tensorflow
!pip install seqeval

In [ ]:
import spacy
spacy.require_cpu()
from spacy.tokens import DocBin
from tqdm import tqdm
import json

from seqeval.metrics import classification_report #to evaluate model
from spacy.tokens import DocBin #to load model

nlp = spacy.blank("en") # load a new spacy model
db = DocBin() # create a DocBin object

In [ ]:
!python -m spacy info


============================== Info about spaCy ==============================

spaCy version    3.8.14                        
Location         /usr/local/lib/python3.12/dist-packages/spacy
Platform         Linux-6.6.122+-x86_64-with-glibc2.35
Python version   3.12.13                       
Pipelines        en_core_web_sm (3.8.0)        



In [ ]:
rtc_train = open('rtc_train_annotations_3.json')
TRAIN_DATA = json.load(rtc_train)

rtc_test = open('rtc_test_annotations_3.json')
TEST_DATA = json.load(rtc_test)

#rtc_eval_test = open('RTC-NER Model/rtc_eval_test_annotations.json')
#TEST_EVAL_DATA = json.load(rtc_eval_test)

In [ ]:
# Create DocBin Objects
for text, annot in tqdm(TRAIN_DATA['annotations']):
    doc = nlp.make_doc(text)
    ents = []
    for start, end, label in annot["entities"]:
        span = doc.char_span(start, end, label=label, alignment_mode="contract")
        if span is None:
            print("Skipping entity")
        else:
            ents.append(span)
    doc.ents = ents
    db.add(doc)

db.to_disk("./training_data.spacy") # save the docbin object

for text, annot in tqdm(TEST_DATA['annotations']):
    doc = nlp.make_doc(text)
    ents = []
    for start, end, label in annot["entities"]:
        span = doc.char_span(start, end, label=label, alignment_mode="contract")
        if span is None:
            print("Skipping entity")
        else:
            ents.append(span)
    doc.ents = ents
    db.add(doc)

db.to_disk("./test_data.spacy") # save the docbin object

  0%|          | 0/1 [00:00<?, ?it/s]

Skipping entity
Skipping entity
Skipping entity
Skipping entity
Skipping entity
Skipping entity
Skipping entity
Skipping entity
Skipping entity
Skipping entity
Skipping entity


100%|██████████| 2/2 [00:00<00:00,  5.72it/s]


Skipping entity
Skipping entity
Skipping entity


In [ ]:
! python -m spacy init config config.cfg --lang en --pipeline ner --optimize efficiency

⚠ To generate a more effective transformer-based config (GPU-only),
install the spacy-transformers package and re-run this command. The config
generated now does not use transformers.
ℹ Generated config template specific for your use case
- Language: en
- Pipeline: ner
- Optimize for: efficiency
- Hardware: CPU
- Transformer: None
✔ Auto-filled config with all values
✔ Saved config
config.cfg
You can now add your data and train your pipeline:
python -m spacy train config.cfg --paths.train ./train.spacy --paths.dev ./dev.spacy


In [ ]:
#! python -m spacy train config.cfg --output ./ --paths.train ./training_data.spacy --paths.dev ./test_data.spacy --gpu-id 0

!python -m spacy train config.cfg \
  --output ./ \
  --paths.train training_data.spacy \
  --paths.dev test_data.spacy

ℹ Saving to output directory: .
ℹ Using CPU

=========================== Initializing pipeline ===========================
✔ Initialized pipeline

============================= Training pipeline =============================
ℹ Pipeline: ['tok2vec', 'ner']
ℹ Initial learn rate: 0.001
E    #       LOSS TOK2VEC  LOSS NER  ENTS_F  ENTS_P  ENTS_R  SCORE 
---  ------  ------------  --------  ------  ------  ------  ------
  0       0          0.00   6261.45    0.00    0.00    0.00    0.00
 50     200     210572.44  696988.31   87.99   82.99   93.63    0.88
100     400      33658.15  97336.84   93.99   89.53   98.91    0.94
150     600      31546.18  57466.55   94.30   90.93   97.94    0.94
200     800      38450.05  50837.50   94.54   90.22   99.29    0.95
250    1000      44800.12  49613.18   94.46   90.66   98.58    0.94
300    1200      48369.90  46537.48   94.52   90.06   99.46    0.95
350    1400      55199.61  45535.58   94.51   90.58   98.79    0.95
400    1600      57517.85  44768.79

## **Evaluation of the rtc_ner_extended model**

In [ ]:
!python -m spacy evaluate 'model-best/' "test_data.spacy"

ℹ Using CPU

================================== Results ==================================

TOK     100.00
NER P   90.22 
NER R   99.29 
NER F   94.54 
SPEED   12356 


=============================== NER (per type) ===============================

                           P        R        F
FACTORS                90.94    98.83    94.72
WEEKDAY                89.90    99.61    94.51
TIME                   87.78   100.00    93.49
PERSON                 91.26    99.29    95.11
NO_VEHICLES            83.21   100.00    90.84
PERSONS_INVOLVED       88.71   100.00    94.02
VEHICLE_TYPE           90.88    99.69    95.08
PLATE_NO               87.81   100.00    93.51
VICTIM_ORGANIZATION    87.69    98.28    92.68
ADDRESS               100.00   100.00   100.00



In [ ]:

#Load the newly creaed model
rtc_ner_ext = spacy.load("model-best")

In [ ]:
doc = rtc_ner_ext(""" no fewer than twenty-five persons have escaped death, been confirmed dead in a multiple accident that happened in anambra state on saturday, july 1, 2023.
the ghastly accident, which happened at the odumodu junction umunya, by nteje-awka expressway, occured at about 12:40pm. the acting sector public education officer, federal road
safety corps (frsc) anambra state, rc margaret onabe, who confirmed this in a press statement to the newsmen, on behalf of the state sector commander, cc adeoye irelewuyi, said that
five vehicles and twenty-six persons were involved in the crash. she said, ?five unidentified drivers were involved in a fatal road traffic crash at odumodu junction umunya, by nteje-awka
expressway, today 1st july, 2023, at about 12:40hrs.? continuing, she gave the names and details of the vehicles, to include toyota hiace (with the registration number enu 32 xd);
mack tanker laden with pms (with the registration number lsd 339 xa); toyota yaris (with the registration number ksf 646 bz); mack tanker (with the registration number bau 305 ze),
and toyota camry (with the registration number abn 64 ja). the frsc spokesperson further noted that the probable cause of the accident was speeding. she said, ?according to eyewitness,
the driver of the truck, with registration number lsd 339 xa, lost control as a result of speed; thereby causing multiple collision of four other vehicles. ?26 people (comprising 20 male adults,
 and 6 female adults) were involved in the crash. ?injured victims were 12 (comprising 9 male adults, and 3 female adults). one male adult was killed while, 13 people were rescued unhurt.?
 while noting that the sector commander, cc irelewuyi was at the scene of the accident with the head of operations, dcc okora awassam; she further hinted that the officers of the anambra
 State Fire Service were also on ground with their firefighting truck in case of fire outbreak. rc onabe added that the frsc team were also on ground controlling traffic and ensuring
 obstruction caused by the crash is cleared, while efforts were being made to remove the trapped body. she said the rescued victims were rushed to the borromeo and divine favour hospital,
 umunya for medical attention. according to her, cc irelewuyi sympathized with the family of the dead victim and wishes the injured victims quick recovery. ?he seriously warned motorists
 to desist from speeding and ensure they drive within minimum safe speed to save their lives and those of other road users,? she said.""")

In [ ]:
spacy.displacy.render(doc, style="ent", jupyter=True) # display in Jupyter

In [ ]:
for ent in doc.ents:
	print(ent.text, " ->>>> ", ent.label_)

multiple  ->>>>  FACTORS
saturday  ->>>>  WEEKDAY
junction  ->>>>  FACTORS
12:40pm.  ->>>>  TIME
rc margaret onabe  ->>>>  PERSON
cc adeoye irelewuyi  ->>>>  PERSON
five vehicles  ->>>>  NO_VEHICLES
twenty-six persons  ->>>>  PERSONS_INVOLVED
junction  ->>>>  FACTORS
12:40hrs.  ->>>>  TIME
toyota hiace  ->>>>  VEHICLE_TYPE
enu 32 xd  ->>>>  TIME
mack tanker  ->>>>  VEHICLE_TYPE
pms  ->>>>  FACTORS
lsd 339 xa  ->>>>  PLATE_NO
toyota yaris  ->>>>  VEHICLE_TYPE
ksf 646 bz  ->>>>  PLATE_NO
mack tanker  ->>>>  VEHICLE_TYPE
bau 305 ze  ->>>>  PLATE_NO
toyota camry  ->>>>  VEHICLE_TYPE
abn 64 ja  ->>>>  PLATE_NO
speeding.  ->>>>  FACTORS
339 xa  ->>>>  PLATE_NO
lost control  ->>>>  FACTORS
speed  ->>>>  FACTORS
multiple collision  ->>>>  FACTORS
26 people  ->>>>  PERSONS_INVOLVED
male adults  ->>>>  FACTORS
female adults  ->>>>  FACTORS
cc irelewuyi  ->>>>  PERSON
dcc okora awassam  ->>>>  PERSON
rc onabe  ->>>>  PERSON
cc irelewuyi  ->>>>  PERSON
